# Presidential speeches -> keyword CSV

Makes `speech_keywords.csv` with one row per speech: president, year, title, speech type, and a count for each keyword.

Run this from the repo root. Each file in `speeches/` is one speech stored as JSON, which Python reads as a dictionary with keys like `president`, `date`, `title` and `transcript`.

In [1]:
import glob
import json
import re

import pandas as pd

## Settings

Change these two to change the CSV. The data has no speech-type field, so the type is guessed from the title: the first phrase below that appears in the title wins.

In [2]:
keywords = ["war", "peace", "freedom", "slavery", "economy", "taxes", "immigration"]

speech_types = {
    "inaugural": "Inaugural Address",
    "fireside chat": "Fireside Chat",
    "state of the union": "State of the Union",
    "annual message": "State of the Union",
    "annual address": "State of the Union",
    "veto": "Veto Message",
    "proclamation": "Proclamation",
    "debate": "Debate",
    "press conference": "Press Conference",
    "news conference": "Press Conference",
    "briefing": "Press Conference",
    "farewell": "Farewell Address",
    "national convention": "Convention Speech",
    "nomination": "Convention Speech",
    "message": "Message",
    "remarks": "Remarks / Statement",
    "statement": "Remarks / Statement",
    "address": "Address / Speech",
    "speech": "Address / Speech",
}

## Go through every speech

In [3]:
rows = []

for filename in sorted(glob.glob("speeches/*.json")):
    with open(filename, encoding="utf-8") as f:
        speech = json.load(f)

    # speech type: the first phrase that appears in the title
    title = speech["title"].lower()
    speech_type = "Other"
    for phrase, label in speech_types.items():
        if phrase in title:
            speech_type = label
            break

    # split the transcript into lowercase words
    text = speech["transcript"].lower()
    text = re.sub(r"<[^>]+>", " ", text)   # remove HTML tags such as <br />
    words = re.findall(r"[a-z]+", text)

    row = {
        "president": speech["president"],
        "year": int(speech["date"][:4]),
        "title": speech["title"],
        "speech_type": speech_type,
    }
    for keyword in keywords:
        row[keyword] = words.count(keyword)

    rows.append(row)

## Save

In [4]:
df = pd.DataFrame(rows)
df.to_csv("speech_keywords.csv", index=False)
df.head()

,president,year,title,speech_type,war,peace,freedom,slavery,economy,taxes,immigration
0,Abraham Lincoln,1852,"July 6, 1852: Eulogy on Henry Clay",Other,14,12,6,30,0,0,0
1,Abraham Lincoln,1854,"October 16, 1854: At Peoria, Illinois",Other,12,18,10,240,0,0,0
2,Abraham Lincoln,1858,"June 16, 1858: ""A House Divided"" Speech",Address / Speech,0,0,3,22,0,0,0
3,Abraham Lincoln,1860,"February 27, 1860: Cooper Union Address",Address / Speech,0,3,3,100,0,0,0
4,Abraham Lincoln,1861,"February 11, 1861: Farewell Address",Farewell Address,0,0,0,0,0,0,0


In [5]:
df["speech_type"].value_counts()

speech_type
State of the Union     210
Address / Speech       204
Message                158
Remarks / Statement    139
Other                   87
Inaugural Address       60
Veto Message            44
Press Conference        39
Proclamation            38
Fireside Chat           30
Convention Speech       24
Farewell Address        13
Debate                  13
Name: count, dtype: int64

## Optional: a more sophisticated version

The above cell is the simplest approach. But we can make it more sophisticated.
The cell below is the same loop again, with two additions marked `# NEW`:

- `word_count`, so a count can be turned into a rate per 1,000 words
- word families: `econom*` counts every word that starts with `econom` (*economy*, *economic*, *economics*...)

It saves a second file, `speech_keywords_extended.csv`. Feel free to compare them.

In [6]:
import html                                # NEW

families = ["econom", "tax", "immigra"]    # NEW

rows = []

for filename in sorted(glob.glob("speeches/*.json")):
    with open(filename, encoding="utf-8") as f:
        speech = json.load(f)

    # speech type: the first phrase that appears in the title
    title = speech["title"].lower()
    speech_type = "Other"
    for phrase, label in speech_types.items():
        if phrase in title:
            speech_type = label
            break

    # split the transcript into lowercase words
    text = speech["transcript"].lower()
    text = re.sub(r"<[^>]+>", " ", text)   # remove HTML tags such as <br />
    text = html.unescape(text)             # NEW: turn codes like &nbsp; into real characters
    words = re.findall(r"[a-z]+", text)

    row = {
        "president": speech["president"],
        "year": int(speech["date"][:4]),
        "title": speech["title"],
        "speech_type": speech_type,
        "word_count": len(words),          # NEW
    }
    for keyword in keywords:
        row[keyword] = words.count(keyword)
    for family in families:                # NEW
        row[family + "*"] = len([word for word in words if word.startswith(family)])

    rows.append(row)

df_extended = pd.DataFrame(rows)
df_extended.to_csv("speech_keywords_extended.csv", index=False)
df_extended.head()

,president,year,title,speech_type,word_count,war,peace,freedom,slavery,economy,taxes,immigration,econom*,tax*,immigra*
0,Abraham Lincoln,1852,"July 6, 1852: Eulogy on Henry Clay",Other,10814,14,12,6,30,0,0,0,0,0,0
1,Abraham Lincoln,1854,"October 16, 1854: At Peoria, Illinois",Other,33240,12,18,10,240,0,0,0,0,0,0
2,Abraham Lincoln,1858,"June 16, 1858: ""A House Divided"" Speech",Address / Speech,3224,0,0,3,22,0,0,0,0,0,0
3,Abraham Lincoln,1860,"February 27, 1860: Cooper Union Address",Address / Speech,13719,0,3,3,100,0,0,0,0,0,0
4,Abraham Lincoln,1861,"February 11, 1861: Farewell Address",Farewell Address,152,0,0,0,0,0,0,0,0,0,0


### Why add a word count?

Some decades have far more text than others, so raw counts mostly show where the most words are. Dividing by the number of words makes decades comparable. Compare the `war` column with the rate next to it.

In [7]:
by_decade = df_extended.groupby(df_extended["year"] // 10 * 10)[["word_count", "war"]].sum()
by_decade["war_per_1000_words"] = by_decade["war"] / by_decade["word_count"] * 1000
by_decade.sort_values("war", ascending=False).head()

,word_count,war,war_per_1000_words
year,,,
1960,422106,770,1.824186
1940,103668,513,4.948489
1840,200734,495,2.465950
1860,187944,412,2.192142
1900,251080,294,1.170942


### Why add word families?

Here's the exact word next to its family, totalled over all speeches. You can see why this matters.

In [8]:
df_extended[["economy", "econom*", "taxes", "tax*", "immigration", "immigra*"]].sum()

economy        1728
econom*        4137
taxes           805
tax*           3408
immigration     360
immigra*        617
dtype: int64

## Keep in mind

- Debates and press conferences also contain the words of moderators, opponents and reporters.
- A word family can catch words you did not mean, so check what it matches before trusting the column.